# Notebook 46 - How much retraining is enough? The recovery-budget curve at 90%

**Why this notebook.** Notebook 40 measured two retraining budgets for the 90% students: one cheap pass over a tenth of the training rows, which leaves the shallow network escalating 12 to 32% of benign traffic and removes three rare attack families on both networks, and two full epochs, which bring every endpoint back to the dense model's range. Between those two points nothing is known: an operator who can afford half an epoch does not know whether the families are back, and the paper's claim that the aggregate risk score passes before the families return rests on two points. This notebook fills the curve.

**What it does.** GPU, about two and a half hours. The committed 90% structures of notebook 40 for magnitude, Taylor and Fisher, and the dense reference, are recovered under five budgets, one pass over a seeded 10%, 25% and 50% subset and one and two full epochs, with three new seeds (601, 701, 809), so the two-epoch point adds three seeds to notebook 40's three. Every model is recalibrated on the fixed slice and audited on the class-balanced evaluation subsample (HSR, benign escalation, attack miss, family and fine macro-F1, per-family recall, ECE with its sign) and, after recalibration, on a 200,000-row natural-prior sample of the validation partition.

**Readings, fixed in stage 2 before any result.** For each network and rule, the smallest budget at which the aggregate passes (paired HSR gap interval below 0.02), at which every family is within 0.15 of the dense reference, and at which benign escalation is within 0.02 of the dense reference; the pre-registered expectation is that on the shallow network the family budget exceeds the aggregate budget for at least one rule. Paired calibration is reported at every budget on both evaluation priors.

In [ ]:
# Stage 1 - bootstrap (GPU runtime)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, gc, time, copy, itertools, subprocess, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
importlib.invalidate_caches()
from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit
from src.saber.surgery import prune_cnn1d_channels, count_parameters
from src.saber.deep_model import DeepCNN1D

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"; N40 = R / "40_compression_headroom"
OUT = R / "46_recovery_budget_curve"; OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = REPO / "models/ciciot2023"
assert (N40 / "N40_structures.csv").exists(), "NB40 results missing; pull the repo"
assert (MODEL_DIR / "deepcnn1d_g5_seed0.pt").exists(), "deep teacher checkpoint missing under models/ciciot2023"
print("device:", DEVICE, "| torch", torch.__version__)

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
PREREG = {
    "notebook": "46_recovery_budget_curve",
    "question": ("at 90% realised MAC reduction on CICIoT2023, how much retraining is needed before a pruned detector matches the "
                 "dense model recovered the same way on the aggregate risk, on every attack family and on benign escalation, and "
                 "does the aggregate pass before the families return"),
    "motivation": ("notebook 40 measured two budgets only: one pass over a 10% subset (shallow: HSR gap +0.039 for the best rule, "
                   "benign escalation 12 to 32%, brute force, malware and web near zero recall; deep: aggregate within the dense "
                   "range but the same families lost for Fisher, Taylor and leverage) and two full epochs (every endpoint within the "
                   "dense range). The curve between them is the operator's decision"),
    "design": {
        "corpus": "CICIoT2023 in the frozen notebook 40 form: same partitions, clipping bounds (0.1/99.9 percentiles of 400,000 training rows), teachers, class weights and evaluation subsample",
        "structures": "the committed notebook 40 90% structures for magnitude, Taylor and Fisher (N40_{arch}_{method}_b90_removed_groups.csv), both networks; the dense reference",
        "recovery_budgets": {"0.10": "one pass over a seeded 10% subset of the training rows (notebook 40's cheap pass)",
                             "0.25": "one pass over a seeded 25% subset", "0.50": "one pass over a seeded 50% subset",
                             "1.00": "one full epoch, seeded order", "2.00": "two full epochs, seeded order (notebook 40's full recovery)"},
        "optimiser": "batch 1,024, Adam 1e-3, cross-entropy with inverse-square-root class weights, as notebook 40",
        "seeds": [601, 701, 809],
        "seeds_note": "new seeds, disjoint from notebook 40's, so the two-epoch point pools to six seeds with notebook 40 for these three rules",
        "reference": "the dense teacher recovered under the same seed, subset and budget, and recalibrated the same way (paired)",
        "recalibration": "every recovered model: BatchNorm statistics re-estimated on the first 51,200 rows of a seed-2026 permutation of the training rows, weights frozen",
        "evaluation": ("primary: the notebook 40 class-balanced evaluation subsample (at most 4,000 rows per class); secondary, "
                       "recalibrated models only: a 200,000-row natural-prior sample of the validation partition (seed 2026)"),
        "endpoints": ("HSR under the balanced profile (primary), benign escalation E, attack miss M, family and fine macro-F1, per-family "
                      "recall, 15-bin equal-mass ECE and its sign (mean confidence minus accuracy)"),
    },
    "thresholds": {"aggregate": 0.02, "family": 0.15, "escalation": 0.02},
    "readings": {
        "R_aggregate": "per network and rule: the smallest budget at which the paired HSR gap's 95% interval lies entirely below 0.02",
        "R_family": "per network and rule: the smallest budget at which every family's mean recall is within 0.15 of the dense reference recovered with the same budget",
        "R_escalation": "per network and rule: the smallest budget at which mean benign escalation is within 0.02 of the dense reference's",
        "C_calibration": "paired ECE (student minus dense) at every budget, balanced subsample and natural-prior sample, with the sign of the dense reference's confidence minus accuracy",
    },
    "expectations": {
        "E1": "on the shallow network R_family > R_aggregate for at least one of the three rules (the aggregate passes before the families return)",
        "E2": "on the deep network R_aggregate = 0.10 for magnitude (notebook 40: the cheap pass already matches the dense reference in the aggregate)",
        "E3": "R_family <= 2.00 for every rule on both networks (two epochs repair every family, notebook 40)",
    },
    "reporting_rule": "every reading reported as measured; a failed expectation changes the paper's text, it is not dropped",
    "no_test_access": True, "no_new_selection": True,
}
pre_path = OUT / "N46_PREREGISTRATION.json"
if pre_path.exists():
    print("pre-registration already exists; left untouched:", pre_path)
else:
    pre_path.write_text(json.dumps(PREREG, indent=2)); print("pre-registration written:", pre_path)
print(json.dumps(PREREG, indent=2))

ARCHS = ["shallow", "deep"]; METHODS = ["magnitude", "taylor", "fisher"]; MODELS = ["dense"] + METHODS
BUDGETS_R = [0.10, 0.25, 0.50, 1.00, 2.00]; SEEDS = PREREG["design"]["seeds"]
TAU = PREREG["thresholds"]; CAL_SEED = 2026; NAT_SEED = 2026; NAT_ROWS = 200_000
MIN_W = {"shallow": int(yaml.safe_load(open(REPO / "config/saber.yaml"))["groups"]["minimum_remaining_per_layer"]), "deep": 8}
NAME = {"magnitude": "magnitude", "taylor": "Taylor", "fisher": "Fisher", "dense": "dense"}

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/46_recovery_budget_curve/N46_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.is_file() and p.name != "N46_PREREGISTRATION.json" for p in OUT.rglob("*"))
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB46 pre-registration, committed before any result: the recovery-budget curve at 90% on CICIoT2023. The committed "
                        "NB40 90% structures (magnitude, Taylor, Fisher) and the dense reference recovered under five budgets (one pass over a "
                        "10%, 25% or 50% seeded subset; one and two full epochs) with three new seeds (601, 701, 809), paired, every model "
                        "recalibrated on the fixed slice, endpoints on the balanced subsample plus a 200k natural-prior sample. Readings: "
                        "smallest budget at which the aggregate passes (gap interval below 0.02), every family is within 0.15 of dense, and "
                        "benign escalation within 0.02; paired ECE at every budget. Expectations E1 (shallow: family budget exceeds aggregate "
                        "budget for some rule), E2 (deep magnitude aggregate passes at 0.10), E3 (two epochs repair every family). "
                        "No test access, no new selection."],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)

In [ ]:
# Stage 3 - data, teachers and helpers (as notebook 40)
TRAIN_LOADER, VAL_LOADER, _T, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES); N_CLASSES = len(CLASS_NAMES)
TEACHERS = {"shallow": SHALLOW_TEACHER.to(DEVICE).eval()}
_dt = DeepCNN1D(N_CLASSES); _dt.load_state_dict(torch.load(MODEL_DIR / "deepcnn1d_g5_seed0.pt", map_location="cpu", weights_only=False)["state_dict"])
TEACHERS["deep"] = _dt.to(DEVICE).eval()

Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy(); Xt, Yt = TRAIN_LOADER.dataset.tensors; N_TRAIN = len(Xt)
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]; EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
_nat = np.random.default_rng(NAT_SEED).permutation(len(Xv))[:NAT_ROWS]
NAT_X = Xv[_nat].to(DEVICE); NAT_Y = VAL_Y_ALL[_nat]
_counts = np.bincount(Yt.numpy(), minlength=N_CLASSES); _w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0]); _w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
CAL_LOADER = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, _cal_idx.tolist()), batch_size=1024, shuffle=False)
_sub = Xt[torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(2026))[:400_000]].numpy()
LO = torch.tensor(np.percentile(_sub, 0.1, axis=0), dtype=torch.float32, device=DEVICE); HI = torch.tensor(np.percentile(_sub, 99.9, axis=0), dtype=torch.float32, device=DEVICE)
del _sub
FAM = np.array(taxonomy.family_by_class); FAMILIES = sorted(set(FAM.tolist())); FAM_Y = FAM[EX_Y]


class Clipped(nn.Module):
    def __init__(self, inner, lo, hi):
        super().__init__(); self.inner = inner; self.register_buffer("lo", lo.clone()); self.register_buffer("hi", hi.clone())
    def forward(self, x):
        return self.inner(torch.maximum(torch.minimum(x, self.hi), self.lo))


def forward_logits(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def forward_logits_batch_stats(model, X):
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone()) for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved: m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]; m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval(); return out


def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)


def audit_logits(lg, y, families=True):
    a = full_model_audit(lg, y, taxonomy, DEFAULT_COST_PROFILES); pr = softmax_np(lg); pred = lg.argmax(axis=1)
    out = {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "family_f1": float(a["family_macro_f1"]),
           "fine_f1": float(a["fine_macro_f1"]), "hsr_balanced": float(a["hsr_balanced_soc"]), "hsr_miss": float(a["hsr_miss_sensitive"]),
           "hsr_fatigue": float(a["hsr_alert_fatigue"]), "ece": float(a["ece15"]), "conf_minus_acc": float(pr.max(axis=1).mean() - (pred == y).mean())}
    if families:
        pf = FAM[pred]; fy = FAM[y]
        out.update({f"recall_{f}": float((pf[fy == f] == f).mean()) for f in FAMILIES if (fy == f).any()})
    return out


def audit(model, mode="eval"):
    lg = forward_logits(model, EX_X) if mode == "eval" else forward_logits_batch_stats(model, EX_X)
    return audit_logits(lg, EX_Y)


def audit_natural(model):
    return audit_logits(forward_logits(model, NAT_X), NAT_Y, families=False)


def recalibrate_bn(model, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(CAL_LOADER):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = 0.1
    model.eval(); return model


def make_loader(seed, fraction):
    gen = torch.Generator().manual_seed(seed)
    if fraction < 1.0:
        sub = torch.randperm(N_TRAIN, generator=gen)[: int(N_TRAIN * fraction)]
        return torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()), batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))
    return torch.utils.data.DataLoader(TRAIN_LOADER.dataset, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def recover(model, loader, epochs):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return model.eval()


def prune_map_of(pairs):
    pm = {}
    for layer, ch in pairs: pm.setdefault(str(layer), []).append(int(ch))
    return {l: sorted(c) for l, c in pm.items()}


def student_for(arch, method):
    if method == "dense":
        return copy.deepcopy(TEACHERS[arch]).to(DEVICE)
    rm = pd.read_csv(N40 / f"N40_{arch}_{method}_b90_removed_groups.csv")
    st, _ = prune_cnn1d_channels(TEACHERS[arch], prune_map_of(zip(rm.module_path.astype(str), rm.channel_index.astype(int))), EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W[arch])
    return st.to(DEVICE)


STRUCT40 = pd.read_csv(N40 / "N40_structures.csv").set_index(["architecture", "method", "budget"])
for arch in ARCHS:
    for m in METHODS:
        n = count_parameters(student_for(arch, m)); ref = int(STRUCT40.loc[(arch, m, 0.9), "parameters"])
        assert n == ref, f"{arch}/{m}: rebuilt 90% structure has {n} parameters, NB40 recorded {ref}"
print("CICIoT2023 ready | train", N_TRAIN, "| balanced eval rows", len(_idx), "| natural-prior rows", len(_nat), "| 90% structures match N40_structures.csv")

In [ ]:
# Stage 4 - the runs: 2 networks x (dense + 3 rules) x 3 seeds x 5 budgets, budget-major so a partial run still gives a curve (resumable)
RUNS = OUT / "N46_runs.csv"
rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
done = {(r["architecture"], r["method"], round(float(r["budget_r"]), 2), int(r["seed"])) for r in rows}


def run_one(arch, method, budget_r, seed):
    torch.manual_seed(seed); np.random.seed(seed); t0 = time.time()
    model = Clipped(student_for(arch, method), LO, HI).to(DEVICE)
    if budget_r < 1.0:
        recover(model, make_loader(seed, budget_r), 1)
    else:
        recover(model, make_loader(seed, 1.0), int(round(budget_r)))
    tr = audit(model); bs = audit(model, "batch"); recalibrate_bn(model); rc = audit(model); nat = audit_natural(model)
    row = {"architecture": arch, "method": method, "budget_r": budget_r, "seed": seed, "parameters": int(count_parameters(model.inner)),
           "batch_b2a": bs["b2a"], "collapse_as_trained": bool(tr["b2a"] > 0.10 and bs["b2a"] <= 0.10), "seconds": round(time.time() - t0, 1)}
    row.update({f"trained_{k}": v for k, v in tr.items()}); row.update({f"recal_{k}": v for k, v in rc.items()}); row.update({f"nat_{k}": v for k, v in nat.items()})
    del model; gc.collect()
    return row


JOBS = [(a, m, b, s) for b in BUDGETS_R for a in ARCHS for s in SEEDS for m in MODELS]
for job in JOBS:
    key = (job[0], job[1], round(float(job[2]), 2), int(job[3]))
    if key in done: continue
    row = run_one(*job); rows.append(row); done.add(key); pd.DataFrame(rows).to_csv(RUNS, index=False)
    print(f"{job[0]:7s} {job[1]:9s} R={job[2]:.2f} s{job[3]}: recal HSR {row['recal_hsr_balanced']:.4f} E {row['recal_b2a']:.4f} M {row['recal_a2b']:.4f} "
          f"famF1 {row['recal_family_f1']:.3f} brute {row['recal_recall_brute_force']:.2f} malware {row['recal_recall_malware']:.2f} web {row['recal_recall_web_application']:.2f} "
          f"ECE {row['recal_ece']:.3f} | nat E {row['nat_b2a']:.4f} ECE {row['nat_ece']:.3f} | {row['seconds']:.0f}s")
print("runs:", len(rows), "of", len(JOBS))

In [ ]:
# Stage 5 - analysis and verdict (rules exactly as pre-registered in stage 2)
runs = pd.read_csv(RUNS)
assert len(runs) == len(JOBS), f"expected {len(JOBS)} runs, found {len(runs)}: finish stage 4 first"
COL = "recal_hsr_balanced"


def series(arch, method, budget_r, col=COL):
    return runs[(runs.architecture == arch) & (runs.method == method) & np.isclose(runs.budget_r, budget_r)].set_index("seed")[col].sort_index()


def paired(arch, method, budget_r, col=COL):
    s = series(arch, method, budget_r, col); d = series(arch, "dense", budget_r, col); common = s.index.intersection(d.index)
    g = (s.loc[common] - d.loc[common]).to_numpy(); n = len(g); hw = stats.t.ppf(0.975, n - 1) * g.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    return {"n": int(n), "gap": float(g.mean()), "gap_lo": float(g.mean() - hw), "gap_hi": float(g.mean() + hw)}


FAM_COLS = [f"recal_recall_{f}" for f in FAMILIES]
METRICS = ["recal_hsr_balanced", "recal_b2a", "recal_a2b", "recal_family_f1", "recal_fine_f1", "recal_ece", "recal_conf_minus_acc", "trained_b2a",
           "nat_b2a", "nat_a2b", "nat_fine_f1", "nat_ece", "nat_conf_minus_acc"] + FAM_COLS
summary = []
for arch in ARCHS:
    for b in BUDGETS_R:
        d = runs[(runs.architecture == arch) & (runs.method == "dense") & np.isclose(runs.budget_r, b)]
        summary.append({"architecture": arch, "method": "dense", "budget_r": b, "n": len(d), **{m: float(d[m].mean()) for m in METRICS}})
        for m in METHODS:
            q = runs[(runs.architecture == arch) & (runs.method == m) & np.isclose(runs.budget_r, b)]
            e = paired(arch, m, b, "recal_ece"); ne = paired(arch, m, b, "nat_ece")
            defic = {f: float(d[f"recal_recall_{f}"].mean() - q[f"recal_recall_{f}"].mean()) for f in FAMILIES}
            summary.append({"architecture": arch, "method": m, "budget_r": b, "n": len(q), **{mm: float(q[mm].mean()) for mm in METRICS}, **paired(arch, m, b),
                            "ece_gap": e["gap"], "ece_gap_lo": e["gap_lo"], "ece_gap_hi": e["gap_hi"], "nat_ece_gap": ne["gap"], "nat_ece_gap_lo": ne["gap_lo"], "nat_ece_gap_hi": ne["gap_hi"],
                            "e_diff": float(q.recal_b2a.mean() - d.recal_b2a.mean()), **{f"deficit_{f}": v for f, v in defic.items()},
                            "max_family_deficit": max(defic.values()), "worst_family": max(defic, key=defic.get)})
SUM = pd.DataFrame(summary); SUM.to_csv(OUT / "N46_summary.csv", index=False)

READ = {}
for arch in ARCHS:
    READ[arch] = {}
    for m in METHODS:
        q = SUM[(SUM.architecture == arch) & (SUM.method == m)].sort_values("budget_r")
        r_agg = next((float(r.budget_r) for r in q.itertuples() if r.gap_hi < TAU["aggregate"]), None)
        r_fam = next((float(r.budget_r) for r in q.itertuples() if r.max_family_deficit <= TAU["family"]), None)
        r_esc = next((float(r.budget_r) for r in q.itertuples() if abs(r.e_diff) <= TAU["escalation"]), None)
        READ[arch][m] = {"R_aggregate": r_agg, "R_family": r_fam, "R_escalation": r_esc,
                         "by_budget": {f"{r.budget_r:.2f}": {"gap": round(float(r.gap), 4), "gap_lo": round(float(r.gap_lo), 4), "gap_hi": round(float(r.gap_hi), 4),
                                                            "E": round(float(r.recal_b2a), 4), "E_diff": round(float(r.e_diff), 4), "max_family_deficit": round(float(r.max_family_deficit), 4),
                                                            "worst_family": r.worst_family, "ece_gap": round(float(r.ece_gap), 4), "ece_gap_lo": round(float(r.ece_gap_lo), 4), "ece_gap_hi": round(float(r.ece_gap_hi), 4),
                                                            "nat_ece_gap": round(float(r.nat_ece_gap), 4), "nat_ece_gap_lo": round(float(r.nat_ece_gap_lo), 4), "nat_ece_gap_hi": round(float(r.nat_ece_gap_hi), 4)} for r in q.itertuples()}}
    dq = SUM[(SUM.architecture == arch) & (SUM.method == "dense")].sort_values("budget_r")
    READ[arch]["dense"] = {f"{r.budget_r:.2f}": {"HSR": round(float(r.recal_hsr_balanced), 4), "E": round(float(r.recal_b2a), 4), "M": round(float(r.recal_a2b), 4),
                                               "ece_balanced": round(float(r.recal_ece), 4), "ece_natural": round(float(r.nat_ece), 4), "conf_minus_acc": round(float(r.recal_conf_minus_acc), 4),
                                               **{f: round(float(getattr(r, f"recal_recall_{f}")), 3) for f in FAMILIES}} for r in dq.itertuples()}


def gt(a, b):
    # "family budget exceeds aggregate budget": the family line is reached later than the aggregate, or never while the aggregate is
    return (b is not None) and (a is None or a > b)


E = {"E1_shallow_family_after_aggregate_some_rule": bool(any(gt(READ["shallow"][m]["R_family"], READ["shallow"][m]["R_aggregate"]) for m in METHODS)),
     "E2_deep_magnitude_aggregate_at_0.10": bool(READ["deep"]["magnitude"]["R_aggregate"] == 0.10),
     "E3_two_epochs_repair_every_family_all_rules": bool(all(READ[a][m]["R_family"] is not None for a in ARCHS for m in METHODS))}
cells_c = SUM[SUM.method != "dense"]
verdict = {"notebook": "46_recovery_budget_curve", "readings": READ, "expectations": E,
           "calibration": {"cells": int(len(cells_c)), "student_better_balanced": int((cells_c.ece_gap_hi < 0).sum()), "student_worse_balanced": int((cells_c.ece_gap_lo > 0).sum()),
                           "student_better_natural": int((cells_c.nat_ece_gap_hi < 0).sum()), "student_worse_natural": int((cells_c.nat_ece_gap_lo > 0).sum())},
           "collapses_as_trained": int(runs.collapse_as_trained.sum()), "runs": int(len(runs)), "prereg": PREREG}
(OUT / "N46_verdict.json").write_text(json.dumps(verdict, indent=2, default=float))

for arch in ARCHS:
    print(f"\n{arch}:")
    for m in METHODS:
        r = READ[arch][m]; print(f"  {NAME[m]:9s} R_aggregate {r['R_aggregate']} | R_family {r['R_family']} | R_escalation {r['R_escalation']}")
        for b, v in r["by_budget"].items():
            print(f"     R={b}: gap {v['gap']:+.4f} [{v['gap_lo']:+.4f}, {v['gap_hi']:+.4f}] E {v['E']:.4f} (diff {v['E_diff']:+.4f}) worst family {v['worst_family']} {v['max_family_deficit']:+.3f} | ECE gap {v['ece_gap']:+.3f} [{v['ece_gap_lo']:+.3f}, {v['ece_gap_hi']:+.3f}] natural {v['nat_ece_gap']:+.3f}")
    print("  dense:", {b: (v["HSR"], v["E"], v["ece_balanced"], v["ece_natural"], v["conf_minus_acc"]) for b, v in READ[arch]["dense"].items()})
print("\nexpectations:", json.dumps(E, indent=1)); print("calibration:", verdict["calibration"]); print("as-trained collapses:", verdict["collapses_as_trained"])

In [ ]:
# Stage 6 - figure (600 dpi, paper names): per-family recall and the paired HSR gap against the recovery budget
SUM = pd.read_csv(OUT / "N46_summary.csv")
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "DejaVu Serif"], "font.size": 7, "axes.labelsize": 7,
                     "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6, "pdf.fonttype": 42, "axes.linewidth": 0.6})
COLR = {"magnitude": "#E69F00", "taylor": "#56B4E9", "fisher": "#009E73", "dense": "#1F1F1F"}; MARK = {"magnitude": "s", "taylor": "^", "fisher": "D", "dense": "o"}
SHOW = ["benign", "brute_force", "malware", "web_application"]; SHOWN = {"benign": "benign recall", "brute_force": "brute force recall", "malware": "malware recall", "web_application": "web recall"}
xs = np.arange(len(BUDGETS_R))
fig, axes = plt.subplots(2, 5, figsize=(7.16, 2.9), gridspec_kw={"wspace": 0.42, "hspace": 0.55})
for i, arch in enumerate(ARCHS):
    ax = axes[i, 0]; ax.axhspan(-TAU["aggregate"], TAU["aggregate"], color="#EEEEEE", zorder=0); ax.axhline(0, color="#B8B8B8", lw=0.6)
    for m in METHODS:
        g = SUM[(SUM.architecture == arch) & (SUM.method == m)].sort_values("budget_r")
        ax.errorbar(xs, g.gap, yerr=[g.gap - g.gap_lo, g.gap_hi - g.gap], color=COLR[m], marker=MARK[m], ms=2.6, lw=0.8, capsize=1.2, label=NAME[m], zorder=3)
    ax.set_ylabel(f"{arch}: paired HSR gap"); ax.set_title("aggregate", fontsize=7, pad=2); ax.set_xticks(xs); ax.set_xticklabels(["0.1", "0.25", "0.5", "1", "2"])
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#E6E6E6", lw=0.5); ax.set_axisbelow(True)
    for j, f in enumerate(SHOW):
        ax = axes[i, j + 1]
        d = SUM[(SUM.architecture == arch) & (SUM.method == "dense")].sort_values("budget_r")
        ax.plot(xs, d[f"recal_recall_{f}"], color=COLR["dense"], marker=MARK["dense"], ms=2.4, lw=0.8, label="dense", zorder=3)
        for m in METHODS:
            g = SUM[(SUM.architecture == arch) & (SUM.method == m)].sort_values("budget_r")
            ax.plot(xs, g[f"recal_recall_{f}"], color=COLR[m], marker=MARK[m], ms=2.4, lw=0.8, label=NAME[m], zorder=3)
        ax.set_ylim(0, 1.05); ax.set_title(SHOWN[f], fontsize=7, pad=2); ax.set_xticks(xs); ax.set_xticklabels(["0.1", "0.25", "0.5", "1", "2"])
        ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#E6E6E6", lw=0.5); ax.set_axisbelow(True)
    if i == 1:
        for ax in axes[i]: ax.set_xlabel("recovery budget (epochs)")
h, l = axes[0, 1].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.savefig(OUT / "N46_recovery_curve.png", dpi=600, bbox_inches="tight"); fig.savefig(OUT / "N46_recovery_curve.pdf", bbox_inches="tight"); plt.show()
print("written ->", OUT)

In [ ]:
# Stage 7 - commit the results (standard commit cell; the message is filled in from the verdict)
v = json.load(open(OUT / "N46_verdict.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/46_recovery_budget_curve/N46_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
parts = []
for arch in ARCHS:
    rr = v["readings"][arch]
    parts.append(f"{arch}: " + "; ".join(f"{NAME[m]} aggregate passes at {rr[m]['R_aggregate']}, families within 0.15 at {rr[m]['R_family']}, escalation within 0.02 at {rr[m]['R_escalation']}" for m in METHODS)
                 + f"; dense E by budget {[rr['dense'][b]['E'] for b in rr['dense']]}, dense ECE balanced/natural at two epochs {rr['dense']['2.00']['ece_balanced']:.3f}/{rr['dense']['2.00']['ece_natural']:.3f}, conf-acc {rr['dense']['2.00']['conf_minus_acc']:+.3f}.")
E = v["expectations"]; c = v["calibration"]
msg = (f"NB46 results (pre-registration {pre}): recovery-budget curve at 90% on CICIoT2023, {v['runs']} recovered models (budgets 0.1/0.25/0.5/1/2 epochs, seeds 601/701/809). "
       + " ".join(parts) + f" Expectations: E1 {'HELD' if E['E1_shallow_family_after_aggregate_some_rule'] else 'NOT HELD'}, E2 {'HELD' if E['E2_deep_magnitude_aggregate_at_0.10'] else 'NOT HELD'}, "
       f"E3 {'HELD' if E['E3_two_epochs_repair_every_family_all_rules'] else 'NOT HELD'}. Paired ECE: student better in {c['student_better_balanced']} / worse in {c['student_worse_balanced']} of {c['cells']} cells on the balanced subsample, "
       f"{c['student_better_natural']} / {c['student_worse_natural']} on the natural-prior sample. As-trained collapses: {v['collapses_as_trained']}.")
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)